In [ ]:
%pip install groq

In [ ]:
import os
os.environ['GROQ_API_KEY'] = "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"

import os
from groq import Groq

client = Groq(
    api_key = os.environ.get("GROQ_API_KEY"),
)

In [ ]:
#Definicao do agente, como sera passado as mensagens, como ser chamado e como sera executado, definindo o que e necessario para cada uma dessa acoes.
class Agent: 
    def __init__(self, client, system):
        self.client = client
        self.system = system
        self.messages = []
        if self.messages is not None:
            self.messages.append ({"role": "system", "content": self.system})

    def __call__(self, message = ""):
        if message:
            self.messages.append({"role": "user", "content": message})
        result = self.execute()
        self.messages.append({"role": "assistant", "content": result})
        return result
    
    def execute(self):
        completion = client.chat.completions.create(
            messages = self.messages,
            model = "llama-3.3-70b-versatile" 
        )
        return completion.choices[0].message.content

In [ ]:
#Realizado a adicao de uma nova acao no prompt que usa a ferramenta de distancia dos planetas, local onde o prompt diz o que o agente fará, como fazer e mostra as ferramentas de pode utilizar, por isso foi adicionado o disy_planet_sun
system_prompt = """
You run in a loop of Thought, Action, PAUSE, Observation.
At the end of the loop you output an Answer
Use Thought to describe your thoughts about the question you have been asked.
Use Action to run one of the actions available to you - then return PAUSE.
Observation will be the result of running those actions.

Your available actions are:

calculate:
e.g. calculate: 4 * 7 / 3
Runs a calculation and returns the number - uses Python so be sure to use floating point syntax if necessary

get_planet_mass:
e.g. get_planet_mass: Earth
returns weight of the planet in kg\

dist_planet_sun:
e.g dist_planet_sun: Earth
Returns the distance of the planet from the Sun in meters.

Example session:

Question: What is the mass of Earth times 2?
Thought: I need to find the mass of Earth
Action: get_planet_mass: Earth
PAUSE 

You will be called again with this:

Observation: 5.972e24

Thought: I need to multiply this by 2
Action: calculate: 5.972e24 * 2
PAUSE

You will be called again with this: 

Observation: 1,1944×10e25

If you have the answer, output it as the Answer.

Answer: The mass of Earth times 2 is 1,1944×10e25.

Now it's your turn:
""".strip()

In [ ]:
#tools

def calculate(operation): 
    return eval(operation)

def get_planet_mass(planet) -> float:
    match planet.lower():
        case "earth":
            return 5.972e24
        case "jupiter":
            return 1.898e27
        case "mars":
            return 6.39e23
        case "mercury":
            return 3.285e23
        case "neptune":
            return 1.024e26
        case "saturn":
            return 5.683e26
        case "uranus":
            return 8.681e25
        case "venus":
            return 4.867e24
        case _:
            return 0.0
        
# Na parte de ferramentas foi adicionado a nova funcao para utilizar as distancias dos planetas, local onde se define todas as funcões a serem utilizadas pelo agente       
def dist_planet_sun(planet):
    match planet.lower():
        case "earth":
            return 1.496e11
        case "jupiter":
            return 7.785e11
        case "mars":
            return 2.279e11
        case "mercury":
            return 5.79e10
        case "neptune":
            return 4.503e12
        case "saturn":
            return 1.433e12
        case "uranus":
            return 2.877e12
        case "venus":
            return 1.082e11
        case _:
            return 0.0


In [ ]:
#Definicao do agente e seu loop, após isso imposto uma condicão de quebra do loop para que seja exibida a resposta esperada
import re

def agent_loop(max_iterations,system, query):
    agent = Agent(client, system_prompt)
    tools = ['calculate', 'get_planet_mass', 'dist_planet_sun']
    next_prompt = query
    i = 0

    while i < max_iterations:
        i += 1
        result = agent(next_prompt)
        print(result)

        if "PAUSE" in result and "Action" in result:
            action = re.findall(r"Action: ([a-z_]+): (.+)", result, re.IGNORECASE)
            chosen_tool = action [0][0]
            arg = action [0][1]

            if chosen_tool in tools:
                result_tool = eval(f"{chosen_tool} ('{arg}')")
                next_prompt = f"Observation: {result_tool}"
            
            else:
                next_prompt = "Observation: Tool not found"

            print(next_prompt)
            continue
        if "Answer" in result:
            break

In [ ]:
neil_tyson = Agent(client, system_prompt)

In [ ]:
agent_loop(max_iterations = 10, system = system_prompt, query = "What is the mass of the Earth plus the mass os Mercury, and all of it time 5?")

In [ ]:
#Na parte de teste foi onde testei todas as funcões e suas limitacões, para identificar o que era necessário adicionar. Teste de comparacao para saber se o agente consegue identificar o planeta mais pesado entre dois selecionados
agent_loop(max_iterations = 10, system = system_prompt, query = "Which planet weighs more, Earth or Mars?")

In [ ]:
#Teste da nova funcao implementada para saber se estava tudo de acordo na parte de ferramentas e com as alteracões do prompt do agente 
agent_loop(max_iterations = 10, system = system_prompt, query = "Which planet is farthest from sun?")

In [ ]:
#Teste para ver se o agente consegue compilar duas funcoes ao mesmo tempo, era esperado duas respostas contendo um planeta cada, foi retornado com sucesso o planeta mais distante e depois o com maior massa
agent_loop(max_iterations = 10, system = system_prompt, query = "Which planet is farthest from sun and which planet is heaviest?")